In [1]:
b_pi = spark.table("bronze.patient_information")

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql.functions import col, trim

pi_trim = b_pi.select([trim(col(c)).alias(c) for c in b_pi.columns])
pi_distinct=pi_trim.distinct()
pi_distinct.count()

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 4, Finished, Available, Finished, False)

64362

In [3]:
dupe_ids = (pi_distinct
            .groupBy("LOG_ID")
            .count()
            .filter("count > 1")
            .select("LOG_ID"))

dupe_rows = pi_distinct.join(dupe_ids, on="LOG_ID", how="left_semi").orderBy("LOG_ID")
#display(dupe_rows)
#display(dupe_rows.select("LOG_ID", "MRN", "IN_OR_DTTM", "OUT_OR_DTTM", "PRIMARY_PROCEDURE_NM").orderBy("LOG_ID"))

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 5, Finished, Available, Finished, False)

In [4]:
from pyspark.sql.functions import row_number, to_json, struct
from pyspark.sql.window import Window
from pyspark.sql.functions import countDistinct

w = Window.partitionBy("LOG_ID").orderBy(to_json(struct(*pi_distinct.columns)))

pi_dedup = (pi_distinct
            .withColumn("rn", row_number().over(w))
            .filter("rn = 1")
            .drop("rn"))

pi_dedup.count()


StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 6, Finished, Available, Finished, False)

64354

In [5]:
pi_dedup.write.mode("overwrite").format("delta").saveAsTable("silver.patient_information")

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 7, Finished, Available, Finished, False)

In [6]:
si = spark.table("silver.patient_information")
print("shape:",si.count(), len(si.columns))

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 8, Finished, Available, Finished, False)

shape: 64354 23


In [7]:
from pyspark.sql.functions import col

SCI = r"^[0-9]+\.?[0-9]*E\+[0-9]+$"

si_flagged = (si
    .withColumn("log_id_corrupt", col("LOG_ID").rlike(SCI))
    .withColumn("mrn_corrupt",    col("MRN").rlike(SCI)))

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 9, Finished, Available, Finished, False)

In [8]:
si_flagged.selectExpr(
    "SUM(CASE WHEN log_id_corrupt THEN 1 ELSE 0 END) AS log_id_bad",
    "SUM(CASE WHEN mrn_corrupt    THEN 1 ELSE 0 END) AS mrn_bad"
).show()

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 10, Finished, Available, Finished, False)

+----------+-------+
|log_id_bad|mrn_bad|
+----------+-------+
|        39|     37|
+----------+-------+



In [9]:
from pyspark.sql.functions import to_timestamp

si_ts = (si_flagged
    .withColumn("in_or",  to_timestamp("IN_OR_DTTM",  "M/d/yy H:mm"))
    .withColumn("out_or", to_timestamp("OUT_OR_DTTM", "M/d/yy H:mm")))

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 11, Finished, Available, Finished, False)

In [10]:
si_ts.selectExpr("MIN(in_or) AS earliest", "MAX(out_or) AS latest").show()

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 12, Finished, Available, Finished, False)

+-------------------+-------------------+
|           earliest|             latest|
+-------------------+-------------------+
|2017-11-12 18:56:00|2023-08-10 12:29:00|
+-------------------+-------------------+



In [11]:
from pyspark.sql.functions import expr

si_dur = si_ts.withColumn(
    "or_duration_min",
    expr("CAST((unix_timestamp(out_or) - unix_timestamp(in_or)) / 60 AS INT)")
)

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 13, Finished, Available, Finished, False)

In [12]:
si_dur.selectExpr(
  "COUNT(or_duration_min) AS n",
  "SUM(CASE WHEN or_duration_min <= 0  THEN 1 ELSE 0 END) AS non_positive",
  "SUM(CASE WHEN or_duration_min < 10  THEN 1 ELSE 0 END) AS under_10",
  "SUM(CASE WHEN or_duration_min > 720 THEN 1 ELSE 0 END) AS over_720",
  "MIN(or_duration_min) AS min_dur",
  "MAX(or_duration_min) AS max_dur",
  "percentile(or_duration_min, 0.01) AS p01",
  "percentile(or_duration_min, 0.50) AS median",
  "percentile(or_duration_min, 0.99) AS p99"
).show()

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 14, Finished, Available, Finished, False)

+-----+------------+--------+--------+-------+-------+----+------+-----+
|    n|non_positive|under_10|over_720|min_dur|max_dur| p01|median|  p99|
+-----+------------+--------+--------+-------+-------+----+------+-----+
|57862|           1|       6|     519|   -979|   1675|38.0| 164.0|707.0|
+-----+------------+--------+--------+-------+-------+----+------+-----+



In [14]:
si_an = (si_dur
    .withColumn("an_start", to_timestamp("AN_START_DATETIME", "M/d/yy H:mm"))
    .withColumn("an_stop",  to_timestamp("AN_STOP_DATETIME",  "M/d/yy H:mm"))
    .withColumn("anes_duration_min",
        expr("CAST((unix_timestamp(an_stop) - unix_timestamp(an_start)) / 60 AS INT)")))

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 16, Finished, Available, Finished, False)

In [15]:
si_an.selectExpr(
  "SUM(CASE WHEN AN_START_DATETIME IS NOT NULL AND AN_START_DATETIME <> '' AND an_start IS NULL THEN 1 ELSE 0 END) AS an_start_fail",
  "SUM(CASE WHEN AN_STOP_DATETIME  IS NOT NULL AND AN_STOP_DATETIME  <> '' AND an_stop  IS NULL THEN 1 ELSE 0 END) AS an_stop_fail",
  "COUNT(CASE WHEN in_or IS NOT NULL AND an_start IS NOT NULL THEN 1 END) AS rows_checked",
  "percentile(ABS(unix_timestamp(in_or) - unix_timestamp(an_start))/60, 0.5)  AS median_gap",
  "percentile(ABS(unix_timestamp(in_or) - unix_timestamp(an_start))/60, 0.99) AS p99_gap"
).show()

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 17, Finished, Available, Finished, False)

+-------------+------------+------------+----------+-------+
|an_start_fail|an_stop_fail|rows_checked|median_gap|p99_gap|
+-------------+------------+------------+----------+-------+
|            0|           0|       55815|       0.0|   10.0|
+-------------+------------+------------+----------+-------+



In [16]:
from pyspark.sql.functions import when, col, expr

si_r1 = (si_an
    .withColumn("out_or_repaired_flag", col("or_duration_min") > 1440)
    .withColumn("out_or",
        when(col("or_duration_min") > 1440, expr("out_or - INTERVAL 1 DAY"))
        .otherwise(col("out_or")))
    .withColumn("or_duration_min",
        expr("CAST((unix_timestamp(out_or) - unix_timestamp(in_or)) / 60 AS INT)")))

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 18, Finished, Available, Finished, False)

In [17]:
si_r1.filter("out_or_repaired_flag") \
     .selectExpr("LOG_ID", "or_duration_min", "anes_duration_min") \
     .show()

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 19, Finished, Available, Finished, False)

+----------------+---------------+-----------------+
|          LOG_ID|or_duration_min|anes_duration_min|
+----------------+---------------+-----------------+
|8cdd6ec802f7666c|             87|             NULL|
|fca5be0ad57cdea5|            114|              120|
|8a289c6bc41711df|            235|              235|
|f5f1addf4628421d|            213|              244|
|33de82b3da6863f8|            207|              226|
|390de95a25b57efa|            138|              146|
|445b2d307b13e01b|             76|             NULL|
|90a6e2859fcb800a|             77|             NULL|
+----------------+---------------+-----------------+



In [18]:
si_r1.selectExpr("MIN(or_duration_min) AS min_dur", "MAX(or_duration_min) AS max_dur").show()

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 20, Finished, Available, Finished, False)

+-------+-------+
|min_dur|max_dur|
+-------+-------+
|   -979|   1399|
+-------+-------+



In [19]:
si_r1.filter("out_or_repaired_flag").count()

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 21, Finished, Available, Finished, False)

8

In [20]:
si_r1.orderBy(col("or_duration_min").desc()) \
     .select("LOG_ID", "IN_OR_DTTM", "OUT_OR_DTTM",
             "or_duration_min", "anes_duration_min",
             "out_or_repaired_flag", "PRIMARY_PROCEDURE_NM") \
     .show(10, truncate=False)

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 22, Finished, Available, Finished, False)

+----------------+--------------+-------------+---------------+-----------------+--------------------+---------------------------------------------------------------+
|LOG_ID          |IN_OR_DTTM    |OUT_OR_DTTM  |or_duration_min|anes_duration_min|out_or_repaired_flag|PRIMARY_PROCEDURE_NM                                           |
+----------------+--------------+-------------+---------------+-----------------+--------------------+---------------------------------------------------------------+
|0483ee1a580022a1|5/16/22 7:15  |5/17/22 6:34 |1399           |1406             |false               |BREAST RECONSTRUCTION, WITH FREE FLAP                          |
|8338d2b6d17feea0|1/17/19 13:23 |1/18/19 12:10|1367           |47               |false               |COLONOSCOPY, DIAGNOSTIC                                        |
|7c2c2b6775222197|2/5/23 7:11   |2/6/23 5:35  |1344           |1371             |false               |LAPAROTOMY, EXPLORATORY, WITH BOWEL RESECTION                  

In [21]:
si_r1.filter("out_or_repaired_flag") \
     .select("LOG_ID", "IN_OR_DTTM", "OUT_OR_DTTM",
             "or_duration_min", "anes_duration_min", "PRIMARY_PROCEDURE_NM") \
     .orderBy("or_duration_min") \
     .show(truncate=False)

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 23, Finished, Available, Finished, False)

+----------------+--------------+--------------+---------------+-----------------+----------------------------------------------------------------+
|LOG_ID          |IN_OR_DTTM    |OUT_OR_DTTM   |or_duration_min|anes_duration_min|PRIMARY_PROCEDURE_NM                                            |
+----------------+--------------+--------------+---------------+-----------------+----------------------------------------------------------------+
|445b2d307b13e01b|9/23/21 16:33 |9/24/21 17:49 |76             |NULL             |MYELOGRAM                                                       |
|90a6e2859fcb800a|3/6/22 14:18  |3/7/22 15:35  |77             |NULL             |INSERTION, DRAIN                                                |
|8cdd6ec802f7666c|2/21/20 12:44 |2/22/20 14:11 |87             |NULL             |ANGIOGRAM, CEREBRAL                                             |
|fca5be0ad57cdea5|2/13/19 10:01 |2/14/19 11:55 |114            |120              |INSERTION, CATHETER, FOR PERIT

In [22]:
si_r1.filter("LOG_ID = '8338d2b6d17feea0'") \
     .select("IN_OR_DTTM", "OUT_OR_DTTM", "AN_START_DATETIME", "AN_STOP_DATETIME",
             "or_duration_min", "anes_duration_min") \
     .show(truncate=False)

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 24, Finished, Available, Finished, False)

+-------------+-------------+-----------------+----------------+---------------+-----------------+
|IN_OR_DTTM   |OUT_OR_DTTM  |AN_START_DATETIME|AN_STOP_DATETIME|or_duration_min|anes_duration_min|
+-------------+-------------+-----------------+----------------+---------------+-----------------+
|1/17/19 13:23|1/18/19 12:10|1/17/19 13:22    |1/17/19 14:09   |1367           |47               |
+-------------+-------------+-----------------+----------------+---------------+-----------------+



In [23]:
si_r1.filter("""
        or_duration_min > 720
        AND anes_duration_min IS NOT NULL
        AND anes_duration_min < 120
     """) \
     .selectExpr("LOG_ID", "or_duration_min", "anes_duration_min",
                 "CAST((unix_timestamp(in_or) - unix_timestamp(an_start))/60 AS INT) AS entry_gap_min",
                 "PRIMARY_PROCEDURE_NM") \
     .orderBy(col("or_duration_min").desc()) \
     .show(50, truncate=False)

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 25, Finished, Available, Finished, False)

+----------------+---------------+-----------------+-------------+-----------------------+
|LOG_ID          |or_duration_min|anes_duration_min|entry_gap_min|PRIMARY_PROCEDURE_NM   |
+----------------+---------------+-----------------+-------------+-----------------------+
|8338d2b6d17feea0|1367           |47               |1            |COLONOSCOPY, DIAGNOSTIC|
+----------------+---------------+-----------------+-------------+-----------------------+



In [24]:
si_r1.filter("""
        anes_duration_min IS NOT NULL
        AND ABS(unix_timestamp(in_or) - unix_timestamp(an_start))/60 > 120
        AND or_duration_min > 3 * anes_duration_min
        AND anes_duration_min >= 60
     """) \
     .selectExpr("LOG_ID", "IN_OR_DTTM", "AN_START_DATETIME",
                 "or_duration_min", "anes_duration_min",
                 "CAST((unix_timestamp(in_or) - unix_timestamp(an_start))/60 AS INT) AS entry_gap_min",
                 "PRIMARY_PROCEDURE_NM") \
     .show(50, truncate=False)

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 26, Finished, Available, Finished, False)

+----------------+-------------+-----------------+---------------+-----------------+-------------+----------------------------------------------------------+
|LOG_ID          |IN_OR_DTTM   |AN_START_DATETIME|or_duration_min|anes_duration_min|entry_gap_min|PRIMARY_PROCEDURE_NM                                      |
+----------------+-------------+-----------------+---------------+-----------------+-------------+----------------------------------------------------------+
|6c4d7607dac1f718|9/28/19 7:05 |9/28/19 17:06    |700            |97               |-601         |GI ENDOSCOPIC ULTRASOUND UPPER                            |
|a361a18022b74d51|8/26/20 11:06|8/26/20 23:07    |846            |132              |-721         |CHOLECYSTECTOMY, LAPAROSCOPIC, WITH CHOLANGIOGRAM         |
|15d7a340edff847f|7/24/20 8:36 |7/24/20 20:34    |899            |225              |-718         |REVISION, INSERTION, OR REMOVAL, VENTRICULAR ASSIST DEVICE|
|c2b283c60bf6bbcb|11/5/20 11:30|11/5/20 15:17    |30

In [25]:
from pyspark.sql.functions import when, col, expr

rule2 = """
    anes_duration_min IS NOT NULL
    AND ABS(unix_timestamp(in_or) - unix_timestamp(an_start))/60 > 120
    AND or_duration_min > 3 * anes_duration_min
    AND anes_duration_min >= 60
"""

si_r2 = (si_r1
    .withColumn("in_or_repaired_flag", expr(rule2))
    .withColumn("in_or", when(col("in_or_repaired_flag"), col("an_start")).otherwise(col("in_or")))
    .withColumn("or_duration_min",
        expr("CAST((unix_timestamp(out_or) - unix_timestamp(in_or))/60 AS INT)")))

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 27, Finished, Available, Finished, False)

In [26]:
si_r2.filter("in_or_repaired_flag") \
     .select("LOG_ID", "or_duration_min", "anes_duration_min", "PRIMARY_PROCEDURE_NM") \
     .show(truncate=False)

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 28, Finished, Available, Finished, False)

+----------------+---------------+-----------------+----------------------------------------------------------+
|LOG_ID          |or_duration_min|anes_duration_min|PRIMARY_PROCEDURE_NM                                      |
+----------------+---------------+-----------------+----------------------------------------------------------+
|6c4d7607dac1f718|99             |97               |GI ENDOSCOPIC ULTRASOUND UPPER                            |
|a361a18022b74d51|125            |132              |CHOLECYSTECTOMY, LAPAROSCOPIC, WITH CHOLANGIOGRAM         |
|15d7a340edff847f|181            |225              |REVISION, INSERTION, OR REMOVAL, VENTRICULAR ASSIST DEVICE|
|c2b283c60bf6bbcb|74             |88               |IR PLCMT GASTROSTOMY TUBE                                 |
+----------------+---------------+-----------------+----------------------------------------------------------+



In [34]:
si_clean = si_r2.filter("LOG_ID NOT IN ('8338d2b6d17feea0', '2713cad9cf971f24')")

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 36, Finished, Available, Finished, False)

In [35]:
si_clean.selectExpr(
    "COUNT(*) AS n",
    "MIN(or_duration_min) AS min_dur",
    "MAX(or_duration_min) AS max_dur"
).show()

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 37, Finished, Available, Finished, False)

+-----+-------+-------+
|    n|min_dur|max_dur|
+-----+-------+-------+
|64352|      3|   1399|
+-----+-------+-------+



In [36]:
from pyspark.sql.functions import upper, trim, regexp_replace, col

si_named = (si_clean
    .withColumn("procedure_nm",
        upper(trim(regexp_replace(col("PRIMARY_PROCEDURE_NM"), r"\s+", " "))))
    .withColumn("is_compound_name",
        col("PRIMARY_PROCEDURE_NM").rlike(r"(?i)\band\b|\bor\b")))

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 38, Finished, Available, Finished, False)

In [37]:
si_named.selectExpr(
    "COUNT(DISTINCT PRIMARY_PROCEDURE_NM) AS raw_names",
    "COUNT(DISTINCT procedure_nm)        AS clean_names",
    "SUM(CASE WHEN is_compound_name THEN 1 ELSE 0 END) AS compound_cases"
).show()

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 39, Finished, Available, Finished, False)

+---------+-----------+--------------+
|raw_names|clean_names|compound_cases|
+---------+-----------+--------------+
|     1768|       1767|          9015|
+---------+-----------+--------------+



In [38]:
print(si_named.filter(col("PRIMARY_PROCEDURE_NM").rlike(r"(?i)\band\b")).count())
print(si_named.filter(col("PRIMARY_PROCEDURE_NM").rlike(r"(?i)\bor\b")).count())

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 40, Finished, Available, Finished, False)

6791
2466


In [39]:
si_final = si_named.withColumnRenamed("BIRTH_DATE", "age_years")

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 41, Finished, Available, Finished, False)

In [41]:
si_final.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("silver.patient_information")


StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 43, Finished, Available, Finished, False)

AnalysisException: [DELTA_SCHEMA_CHANGE_SINCE_ANALYSIS] The schema of your Delta table has changed in an incompatible way since your DataFrame
or DeltaTable object was created. Please redefine your DataFrame or DeltaTable object.
Changes:
Latest schema is missing field(s): BIRTH_DATE
Latest schema has additional field(s): procedure_nm, out_or_repaired_flag, anes_duration_min, age_years, or_duration_min, out_or, in_or_repaired_flag, an_start, log_id_corrupt, an_stop, is_compound_name, mrn_corrupt, in_or

In [42]:
sp = spark.table("silver.patient_information")
print(sp.count(), len(sp.columns))
sp.printSchema()

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 44, Finished, Available, Finished, False)

64352 35
root
 |-- LOG_ID: string (nullable = true)
 |-- MRN: string (nullable = true)
 |-- DISCH_DISP_C: string (nullable = true)
 |-- DISCH_DISP: string (nullable = true)
 |-- HOSP_ADMSN_TIME: string (nullable = true)
 |-- HOSP_DISCH_TIME: string (nullable = true)
 |-- LOS: string (nullable = true)
 |-- ICU_ADMIN_FLAG: string (nullable = true)
 |-- SURGERY_DATE: string (nullable = true)
 |-- age_years: string (nullable = true)
 |-- HEIGHT: string (nullable = true)
 |-- WEIGHT: string (nullable = true)
 |-- SEX: string (nullable = true)
 |-- PRIMARY_ANES_TYPE_NM: string (nullable = true)
 |-- ASA_RATING_C: string (nullable = true)
 |-- ASA_RATING: string (nullable = true)
 |-- PATIENT_CLASS_GROUP: string (nullable = true)
 |-- PATIENT_CLASS_NM: string (nullable = true)
 |-- PRIMARY_PROCEDURE_NM: string (nullable = true)
 |-- IN_OR_DTTM: string (nullable = true)
 |-- OUT_OR_DTTM: string (nullable = true)
 |-- AN_START_DATETIME: string (nullable = true)
 |-- AN_STOP_DATETIME: string (nu

In [44]:
%%sql
SELECT
    COUNT(*) AS n,
    SUM(CASE WHEN age_years RLIKE '^[0-9]+$' THEN 1 ELSE 0 END)            AS age_int_shaped,
    SUM(CASE WHEN age_years IS NULL OR age_years = '' THEN 1 ELSE 0 END)   AS age_blank,
    SUM(CASE WHEN LOS RLIKE '^[0-9]+(\\.[0-9]+)?$' THEN 1 ELSE 0 END)      AS los_num_shaped,
    SUM(CASE WHEN LOS IS NULL OR LOS = '' THEN 1 ELSE 0 END)               AS los_blank,
    SUM(CASE WHEN ASA_RATING_C IS NULL OR ASA_RATING_C = '' THEN 1 ELSE 0 END) AS asa_blank
FROM silver.patient_information

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 46, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 6 fields>

In [45]:
%%sql
SELECT ICU_ADMIN_FLAG, COUNT(*) AS n
FROM silver.patient_information
GROUP BY 1 ORDER BY 2 DESC

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 47, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

In [46]:
from pyspark.sql.functions import col, hour

si_typed = (si_final
    .withColumn("age_years",    col("age_years").cast("int"))
    .withColumn("los_days",     col("LOS").cast("double"))
    .withColumn("asa_rating_c", col("ASA_RATING_C").cast("int"))
    .withColumn("icu_admit",    col("ICU_ADMIN_FLAG") == "Yes")
    .withColumn("start_hour",   hour(col("in_or"))))

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 48, Finished, Available, Finished, False)

In [47]:
si_typed.selectExpr(
    "SUM(CASE WHEN age_years    IS NULL THEN 1 ELSE 0 END) AS age_null",
    "SUM(CASE WHEN los_days     IS NULL THEN 1 ELSE 0 END) AS los_null",
    "SUM(CASE WHEN asa_rating_c IS NULL THEN 1 ELSE 0 END) AS asa_null",
    "MIN(age_years) AS age_min", "MAX(age_years) AS age_max",
    "MIN(los_days)  AS los_min", "MAX(los_days)  AS los_max",
    "MIN(start_hour) AS hr_min", "MAX(start_hour) AS hr_max"
).show()

StatementMeta(, 42b7de6c-4cb2-4d09-9709-4092aca730ef, 49, Finished, Available, Finished, False)

AnalysisException: [DELTA_SCHEMA_CHANGE_SINCE_ANALYSIS] The schema of your Delta table has changed in an incompatible way since your DataFrame
or DeltaTable object was created. Please redefine your DataFrame or DeltaTable object.
Changes:
Latest schema is missing field(s): BIRTH_DATE
Latest schema has additional field(s): procedure_nm, out_or_repaired_flag, anes_duration_min, age_years, or_duration_min, out_or, in_or_repaired_flag, an_start, log_id_corrupt, an_stop, is_compound_name, mrn_corrupt, in_or